#Importing Libraries



In [1]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

# Loading and Cleaning the Dataset

In [2]:
df = pd.read_excel('data/raw/data_million_.xlsx', engine='calamine')
print(df.shape)

(38124, 15)


In [3]:
df.head()

,Unnamed: 0,Company name Latin alphabet,Country ISO code,Consolidation code,Listing status,Operating revenue (Turnover) m USD 2025,Operating revenue (Turnover) m USD 2024,Operating revenue (Turnover) m USD 2023,Operating revenue (Turnover) m USD 2022,Operating revenue (Turnover) m USD 2021,Total assets m USD 2025,Total assets m USD 2024,Total assets m USD 2023,Total assets m USD 2022,Total assets m USD 2021
0,1.0,TRAFIGURA CONTROL HOLDINGS PTE. LTD.,SG,C2,Unlisted,240268.000,243202.000000,244280.000,318481.100000,231309.600000,79517.900,76943.800000,86505.100,98202.000000,90664.000000
1,2.0,TRAFIGURA GROUP PTE. LTD.,SG,C2,Unlisted,240268.000,243202.000000,244280.000,318481.100000,231309.600000,79493.900,76426.500000,83382.500,98066.300000,90065.700000
2,3.0,NVIDIA SINGAPORE PTE LTD,SG,U1,Unlisted,117774.690,66577.956000,43181.093,20786.815000,21612.757000,26896.486,11333.790000,6628.668,4153.001000,4294.814000
3,4.0,VITOL ASIA PTE LTD,SG,U1,Unlisted,93685.884,85250.900163,83465.374,118286.971245,70743.858449,9700.790,10935.822481,11426.306,12518.357473,13067.135194
4,5.0,WILMAR INTERNATIONAL LIMITED,SG,C1,Listed,70704.083,67772.985000,67715.457,73856.343000,66070.441000,65641.775,59572.170000,61808.692,60401.970000,58718.446000


In [4]:
# Checking for missing values
rev_cols = [col for col in df.columns if 'Operating revenue' in col]
asset_cols = [col for col in df.columns if 'Total assets' in col]
all_metric_cols = rev_cols + asset_cols

# Replace exact zeros (numeric or string) or common text placeholders with NaN
for col in all_metric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce') # Converts text placeholders to NaN
    df[col] = df[col].replace(0.0, np.nan)            # Converts numeric 0 to NaN

# Drop companies where ALL metric columns are missing because we need data 2021 through 2025.
df = df.dropna(subset=all_metric_cols, how='all')

# Display the remaining missing value summary
print(df.isna().sum())


Unnamed: 0                                     0
Company name Latin alphabet                    0
Country ISO code                               0
Consolidation code                             0
Listing status                                 0
Operating revenue (Turnover) m USD 2025    11176
Operating revenue (Turnover) m USD 2024    10728
Operating revenue (Turnover) m USD 2023    10586
Operating revenue (Turnover) m USD 2022     6368
Operating revenue (Turnover) m USD 2021     6824
Total assets m USD 2025                      342
Total assets m USD 2024                      250
Total assets m USD 2023                      225
Total assets m USD 2022                      186
Total assets m USD 2021                      212
dtype: int64


In [5]:
print(df.shape)

(37891, 15)


In [6]:
#Checking if ISO code is SG for all entries
(df['Country ISO code'] != "SG").sum()

np.int64(0)

In [7]:
#Number of public vs pvt firms and removing delisted firms
print(df['Listing status'].value_counts())
df['Listing status'] = df['Listing status'].replace('Delisted', np.nan)
df = df.dropna(subset=['Listing status'])
df['Listing status'].value_counts()

Listing status
Unlisted    37192
Listed        490
Delisted      209
Name: count, dtype: int64


Listing status
Unlisted    37192
Listed        490
Name: count, dtype: int64

• C1 (Consolidated without companion): Consolidated financial statements (mother company integrating subsidiaries and branches) where no separate unconsolidated companion statement is available in Orbis.

• C2 (Consolidated with companion): Consolidated financial statements (mother company integrating subsidiaries and branches) where a separate unconsolidated companion statement is also available in Orbis.

• U1 (Unconsolidated without companion): Unconsolidated financial statements (showing only the single firm, not integrating subsidiaries or branches) where no separate consolidated companion statement is available in Orbis.

• U2 (Unconsolidated with companion): Unconsolidated financial statements (showing only the single firm) where a separate consolidated companion statement is also available in Orbis.


In [8]:
#Keep C1 and U1 and drop C2 and U2
df = df[df['Consolidation code'].isin(['C1', 'U1'])]
df['Consolidation code'].value_counts()

Consolidation code
U1    33231
C1      481
Name: count, dtype: int64

In [9]:
#Checking for duplicate names
print(f'Number of duplicated firms in the dataset: {df['Company name Latin alphabet'].duplicated().sum()}')
display(df[df['Company name Latin alphabet'].duplicated()])

Number of duplicated firms in the dataset: 5


,Unnamed: 0,Company name Latin alphabet,Country ISO code,Consolidation code,Listing status,Operating revenue (Turnover) m USD 2025,Operating revenue (Turnover) m USD 2024,Operating revenue (Turnover) m USD 2023,Operating revenue (Turnover) m USD 2022,Operating revenue (Turnover) m USD 2021,Total assets m USD 2025,Total assets m USD 2024,Total assets m USD 2023,Total assets m USD 2022,Total assets m USD 2021
482,483.0,SWIRE SHIPPING PTE. LTD.,SG,U1,Unlisted,1058.040000,1125.704000,1144.867887,1326.825117,804.487153,809.051000,830.454000,891.106185,1051.115757,961.176911
2912,2913.0,GLAXOCHEM PTE LTD,SG,U1,Unlisted,62.547850,67.109604,58.965344,16.041987,0.069857,1534.893431,1373.146825,1326.274508,1201.382303,1319.511546
6440,6441.0,YHS INTERNATIONAL PTE. LTD.,SG,U1,Unlisted,13.659175,14.008863,12.567645,15.049989,20.080872,71.119357,68.093919,64.432271,60.509379,59.671061
11156,11157.0,BRIDGETEK PTE. LTD.,SG,U1,Unlisted,3.794148,3.926229,5.986941,21.300957,6.671240,11.291730,12.302155,14.434448,13.702251,10.114960
36765,36766.0,EMERALD ENTERPRISE PTE. LTD.,SG,U1,Unlisted,NaN,NaN,NaN,NaN,NaN,227.961078,143.232941,0.000153,0.002953,NaN


In [10]:
name = "Company name Latin alphabet"
# Same-name pairs on U1 with near-identical values: keep first, log the drop
df = df.drop_duplicates(subset=[name], keep="first")
print(f'Number of duplicated firms in the dataset: {df[name].duplicated().sum()}')

Number of duplicated firms in the dataset: 0


In [11]:
# Save the cleaned company-level dataset.
df.to_csv("data/processed/companies_cleaned_wide.csv",
    index=False
)

## Wide to Long

In [12]:
# Reshape the dataframe into a long format (one row per firm per year)
# Columns are in the format: 'Metric name bn USD YYYY'
data_long = pd.wide_to_long(
    df,
    stubnames=['Operating revenue (Turnover) m USD', 'Total assets m USD'],
    i=['Company name Latin alphabet', 'Country ISO code', 'Listing status', 'Consolidation code'],
    j='Year',
    sep=' '
).reset_index()

# Sort the values logically by company and year
data_long = data_long.sort_values(by=['Unnamed: 0', 'Year'])

# Display the long format dataframe
display(data_long.head())

,Company name Latin alphabet,Country ISO code,Listing status,Consolidation code,Year,Unnamed: 0,Operating revenue (Turnover) m USD,Total assets m USD
4,NVIDIA SINGAPORE PTE LTD,SG,Unlisted,U1,2021,3.0,21612.757,4294.814
3,NVIDIA SINGAPORE PTE LTD,SG,Unlisted,U1,2022,3.0,20786.815,4153.001
2,NVIDIA SINGAPORE PTE LTD,SG,Unlisted,U1,2023,3.0,43181.093,6628.668
1,NVIDIA SINGAPORE PTE LTD,SG,Unlisted,U1,2024,3.0,66577.956,11333.790
0,NVIDIA SINGAPORE PTE LTD,SG,Unlisted,U1,2025,3.0,117774.690,26896.486


In [13]:
data_long.shape

(168535, 8)

In [14]:
# Save the final company-year dataset used for tests and charts.
data_long.to_csv("data/processed/company_year_panel.csv",
    index=False
)

## Creating Public vs Pvt Firms

In [15]:
#Grouping by public and pvt companies
public = data_long[data_long['Listing status'] == 'Listed']
private = data_long[data_long['Listing status'] == 'Unlisted']
print(f'Number of public firms across 5 years: {public.shape}')
print(f'Number of private firms across 5 year: {private.shape}')

Number of public firms across 5 years: (2395, 8)
Number of private firms across 5 year: (166140, 8)


For each of the five years, separate firms into public and private groups and create valid revenue and asset samples. Report the sample size for each group and measure.

In [16]:
def summarise(df, col):
    s = df[col]
    return {
        'total': len(s),
        'missing': s.isna().sum(),
        'zero': (s == 0).sum(),
        'negative': (s < 0).sum(),
        'positive': (s > 0).sum(),
    }

rows = []
for y in range(2021, 2026):
    for label, df in [('Public', public), ('Private', private)]:
        d = df[df['Year'] == y]
        for var in ['Operating revenue (Turnover) m USD', 'Total assets m USD']:
            r = summarise(d, var)
            rows.append({'Year': y, 'Group': label, 'Variable': var, **r})

checks = pd.DataFrame(rows)

# Column totals across all rows
count_cols = ['missing', 'zero', 'negative', 'positive']
totals = checks[count_cols].sum()
total_row = pd.DataFrame([{'Year': 'Total', 'Group': '', 'Variable': '', **totals.to_dict()}])
checks_with_total = pd.concat([checks, total_row], ignore_index=True)

display(checks_with_total)

,Year,Group,Variable,total,missing,zero,negative,positive
0,2021,Public,Operating revenue (Turnover) m USD,479.0,2,0,0,477
1,2021,Public,Total assets m USD,479.0,0,0,0,479
2,2021,Private,Operating revenue (Turnover) m USD,33228.0,6267,0,155,26806
3,2021,Private,Total assets m USD,33228.0,207,0,6,33015
4,2022,Public,Operating revenue (Turnover) m USD,479.0,1,0,0,478
5,2022,Public,Total assets m USD,479.0,0,0,0,479
6,2022,Private,Operating revenue (Turnover) m USD,33228.0,5932,0,307,26989
7,2022,Private,Total assets m USD,33228.0,179,0,7,33042
8,2023,Public,Operating revenue (Turnover) m USD,479.0,1,0,0,478
9,2023,Public,Total assets m USD,479.0,0,0,0,479


* all 1,013 negative values are in the private group.
* Total assets: Negative values are almost certainly data errors. Under standard accounting, assets can’t be negative, so dropping them is reasonable. (31 entries)
* Operating revenue (982 negatives): Dropping them is reasonable for a size comparison, since negative turnover doesn’t measure firm size.
* Affected observations are about 0.5–0.9% of each year’s private sample, so the effect on medians should be small.



In [17]:
def clean(df, col, keep_negatives=False):
    s = df[col].dropna()
    return s if keep_negatives else s[s >= 0]

# Main spec: drop negatives
pub_rev  = clean(public[public['Year'] == 2025], 'Operating revenue (Turnover) m USD')
priv_rev = clean(private[private['Year'] == 2025], 'Operating revenue (Turnover) m USD')

# Robustness: keep negatives
pub_rev_all  = clean(public[public['Year'] == 2025], 'Operating revenue (Turnover) m USD', keep_negatives=True)
priv_rev_all = clean(private[private['Year'] == 2025], 'Operating revenue (Turnover) m USD', keep_negatives=True)

print("Median (drop neg):", pub_rev.median(), round(priv_rev.median(),4))
print("Median (keep neg):", pub_rev_all.median(), round(priv_rev_all.median(),4))

Median (drop neg): 63.288 1.752
Median (keep neg): 63.288 1.7158


In [18]:
rev = 'Operating revenue (Turnover) m USD'
id_col = 'Unnamed: 0'

# Firm-years with negative revenue, private group only
neg = private.loc[private[rev] < 0, [id_col, 'Year', rev]]

# 1. How many distinct firms have any negative revenue?
print("Firms with negative revenue in any year:", neg[id_col].nunique())
print("Negative firm-years:", len(neg))

# 2. How many years is each firm negative?
years_per_firm = neg.groupby(id_col)['Year'].nunique()
print("\nNumber of firms by count of negative years:")
print(years_per_firm.value_counts().sort_index())

# 3. Firm x year grid: 1 = negative that year (shows persistence visually)
grid = (neg.assign(neg=1)
           .pivot_table(index=id_col, columns='Year', values='neg',
                        aggfunc='max', fill_value=0)
           .astype(int))
display(grid.sort_values(list(grid.columns), ascending=False).head(30))

# 4. Are negatives persistent across consecutive years?
flags = grid.reindex(columns=range(2021, 2026), fill_value=0)
consecutive = ((flags.iloc[:, :-1].values == 1) & (flags.iloc[:, 1:].values == 1)).any(axis=1)
print("\nFirms negative in at least two consecutive years:", consecutive.sum())

Firms with negative revenue in any year: 596
Negative firm-years: 982

Number of firms by count of negative years:
Year
1    351
2    138
3     77
4     26
5      4
Name: count, dtype: int64


Year,2021,2022,2023,2024,2025
Unnamed: 0,,,,,
37974.0,1,1,1,1,1
37979.0,1,1,1,1,1
38061.0,1,1,1,1,1
38091.0,1,1,1,1,1
11561.0,1,1,1,1,0
20802.0,1,1,1,1,0
26176.0,1,1,1,1,0
37970.0,1,1,1,0,1
38001.0,1,1,1,0,1



Firms negative in at least two consecutive years: 196


In [19]:
rows = []
for y in range(2021, 2026):
    p = public[public['Year'] == y]
    q = private[private['Year'] == y]
    rows.append({
        'Year': y,
        'Public firms': len(p),
        'Public valid revenue': (p['Operating revenue (Turnover) m USD'] > 0).sum(),
        'Public valid assets': (p['Total assets m USD'] > 0).sum(),
        'Private firms': len(q),
        'Private valid revenue': (q['Operating revenue (Turnover) m USD'] > 0).sum(),
        'Private valid assets': (q['Total assets m USD'] > 0).sum(),
    })

sample_sizes = pd.DataFrame(rows)
display(sample_sizes)

,Year,Public firms,Public valid revenue,Public valid assets,Private firms,Private valid revenue,Private valid assets
0,2021,479,477,479,33228,26806,33015
1,2022,479,478,479,33228,26989,33042
2,2023,479,478,479,33228,23245,33010
3,2024,479,479,479,33228,23122,32987
4,2025,479,479,479,33228,22719,32902


# KS Test

For Revenue:

*   Null hypothesis (H₀): The distribution of revenue is the same for public and private firms in that year. $$F_{\text{public}}(x) = F_{\text{private}}(x) \quad \text{for all } x$$
*   Alternative hypothesis (H₁): The distributions differ. Formally,$$F_{\text{public}}(x) \neq F_{\text{private}}(x) \quad \text{for at least one value of } x$$


For Total Assets:

*   Null hypothesis (H₀): The distribution of total assets is the same for public and private firms in that year. Formally, $$F_{\text{public}}(x) = F_{\text{private}}(x) \quad \text{for all } x$$

*   Alternative hypothesis (H₁): The distributions differ. Formally, $$F_{\text{public}}(x) \neq F_{\text{private}}(x) \quad \text{for at least one value of } x$$

In [20]:
measures = {
    'Operating revenue (Turnover) m USD': 'Revenue',
    'Total assets m USD': 'Total assets',
}

ks_results = []
for year in range(2021, 2026):
    for col, label in measures.items():
        # Valid values only: strictly positive, non-missing (NaN > 0 is False)
        pub = public.loc[(public['Year'] == year) & (public[col] > 0), col].to_numpy()
        pri = private.loc[(private['Year'] == year) & (private[col] > 0), col].to_numpy()

        res = ks_2samp(pub, pri)  # two-sided by default
        ks_results.append({
            'Year': year,
            'Measure': label,
            'n_public': len(pub),
            'n_private': len(pri),
            'D': round(res.statistic, 4),
            'p_value': res.pvalue,
        })

ks_table = pd.DataFrame(ks_results)

# Save the unrounded results for the final report.
ks_table.to_csv("data/processed/ks_table.csv",index=False)

display(ks_table)

,Year,Measure,n_public,n_private,D,p_value
0,2021,Revenue,477,26806,0.6467,6.940420e-192
1,2021,Total assets,479,33015,0.6327,1.062517e-183
2,2022,Revenue,478,26989,0.6394,2.513027e-187
3,2022,Total assets,479,33042,0.6105,1.976604e-169
4,2023,Revenue,478,23245,0.5977,2.767334e-160
5,2023,Total assets,479,33010,0.6004,3.057818e-163
6,2024,Revenue,479,23122,0.5942,1.540421e-158
7,2024,Total assets,479,32987,0.5879,8.934752e-156
8,2025,Revenue,479,22719,0.5791,9.816681e-150
9,2025,Total assets,479,32902,0.5819,2.723755e-152


Every p-value is far below 0.001, so the null is rejected for all 10 tests. The D values are large, around 0.58 to 0.65, which means the public and private distributions are very different. They are also declining slightly over time, so the gap is narrowing, but it remains large.

Because the sample sizes are so unequal (about 479 public firms versus 23,000 to 33,000 private firms), the p-values are very small, and the test will detect even minor differences.

# Bootstrap Annual Median Difference `D`

In [21]:
B = 10_000        # bootstrap replications
SEED = 2026         # fixed seed for reproducibility
CHUNK = 200       # resample in blocks to keep memory bounded
rng = np.random.default_rng(SEED)

measures = {
    'Operating revenue (Turnover) m USD': 'Revenue',
    'Total assets m USD': 'Total assets',
}

boot_rows = []
for year in range(2021, 2026):
    for col, label in measures.items():
        # Valid values only: strictly positive, non-missing
        a = public.loc[(public['Year'] == year) & (public[col] > 0), col].to_numpy()
        b = private.loc[(private['Year'] == year) & (private[col] > 0), col].to_numpy()

        point = np.median(a) - np.median(b)

        # Resample each group with replacement, then take the median difference
        boot = np.empty(B)
        for start in range(0, B, CHUNK):
            k = min(CHUNK, B - start)
            boot_a = a[rng.integers(0, len(a), size=(k, len(a)))]
            boot_b = b[rng.integers(0, len(b), size=(k, len(b)))]
            boot[start:start + k] = np.median(boot_a, axis=1) - np.median(boot_b, axis=1)

        ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

        boot_rows.append({
            'Year': year,
            'Measure': label,
            'n_public': len(a),
            'n_private': len(b),
            'M_public': np.median(a),
            'M_private': np.median(b),
            'Difference': point,
            'CI_low_95': ci_low,
            'CI_high_95': ci_high,
        })

boot_table = pd.DataFrame(boot_rows)

# Save the unrounded results for charts and the final report
boot_table.to_csv("data/processed/boot_table.csv",index=False)

display(boot_table.round(4))

,Year,Measure,n_public,n_private,M_public,M_private,Difference,CI_low_95,CI_high_95
0,2021,Revenue,477,26806,57.3182,1.0332,56.2850,46.0276,70.6788
1,2021,Total assets,479,33015,111.5177,2.3388,109.1789,87.4137,145.3780
2,2022,Revenue,478,26989,64.9236,1.1986,63.7250,52.7951,75.6246
3,2022,Total assets,479,33042,110.0136,2.6245,107.3890,89.1750,137.3991
4,2023,Revenue,478,23245,64.3275,1.6061,62.7214,52.3847,80.1035
5,2023,Total assets,479,33010,104.5200,2.7255,101.7945,86.6938,133.8436
6,2024,Revenue,479,23122,63.1302,1.6330,61.4972,49.8970,75.6470
7,2024,Total assets,479,32987,105.7340,2.6560,103.0781,84.1621,135.7391
8,2025,Revenue,479,22719,63.2880,1.7520,61.5360,52.8684,83.1512
9,2025,Total assets,479,32902,118.9756,2.6853,116.2903,91.6204,138.1315


Every confidence interval is above zero, so the public median is larger than the private median in every year and for both measures. The gap is about 60 million USD for revenue and about 100 million USD for total assets, and it is stable across years.

These medians are for firms with positive values only. The private median is very small because many private firms are tiny, so the medians are sensitive to the zero-value exclusion. Note this in the report.